In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
class VariationalAutoencoder(nn.Module):
    def __init__(self, latentDim=32):
        super(VariationalAutoencoder, self).__init__()
        
        # Encoder Architecture
        self.encoderConv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1), # Output: 14x14
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # Output: 7x7
            nn.ReLU(),
            nn.Flatten()
        )
        
        self.fcMu = nn.Linear(64 * 7 * 7, latentDim)
        self.fcLogVar = nn.Linear(64 * 7 * 7, latentDim)
        
        # Decoder Architecture
        self.decoderFc = nn.Linear(latentDim, 64 * 7 * 7)
        self.decoderConv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # Output: 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1), # Output: 28x28
            nn.Sigmoid() # Bounds continuous values to [0, 1] matching normalized data
        )

    def encode(self, x):
        features = self.encoderConv(x)
        mu = self.fcMu(features)
        logVar = self.fcLogVar(features)
        return mu, logVar

    def reparameterize(self, mu, logVar):
        std = torch.exp(0.5 * logVar)
        epsilon = torch.randn_like(std)
        return mu + epsilon * std

    def decode(self, z):
        xFlattened = self.decoderFc(z)
        xReshaped = xFlattened.view(-1, 64, 7, 7)
        return self.decoderConv(xReshaped)

    def forward(self, x):
        mu, logVar = self.encode(x)
        z = self.reparameterize(mu, logVar)
        xHat = self.decode(z)
        return xHat, mu, logVar


def computeLoss(xHat, x, mu, logVar):
    # Gaussian Likelihood with an assumed constant variance reduces directly to Summed Squared Error
    reconLoss = nn.functional.mse_loss(xHat, x, reduction='sum')
    
    # Closed-form KL Divergence for an analytical Gaussian prior N(0, I)
    klLoss = -0.5 * torch.sum(1 + logVar - mu.pow(2) - logVar.exp())
    
    totalLoss = reconLoss + klLoss
    return totalLoss, reconLoss, klLoss


def trainModel():
    # Use Metal Performance Shaders (MPS) for MacBook M4 acceleration
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Target execution hardware configured to: {device}")
    
    # Configuration Hyperparameters
    latentDim = 32
    batchSize = 256
    learningRate = 1e-3
    epochs = 30
    
    # EMNIST Pipeline Setup
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    
    # "balanced" split contains equal distribution of digits and uppercase/lowercase letters
    trainDataset = datasets.EMNIST(
        root="./data", 
        split="balanced", 
        train=True, 
        download=True, 
        transform=transform
    )
    trainLoader = DataLoader(trainDataset, batchSize=batchSize, shuffle=True)
    
    # Explicit Instantiations
    model = VariationalAutoencoder(latentDim=latentDim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learningRate)
    
    # Explicit Manual Training Optimization Loop
    for epoch in range(epochs):
        model.train()
        totalLoss = 0.0
        totalRecon = 0.0
        totalKl = 0.0
        
        for batchIdx, (data, _) in enumerate(trainLoader):
            data = data.to(device)
            
            # Clear historical tracking tensors manually
            optimizer.zero_grad()
            
            # Direct forward execution pass
            reconData, mu, logVar = model(data)
            
            # Compute individual mathematical elements
            loss, reconLoss, klLoss = computeLoss(reconData, data, mu, logVar)
            
            # Direct backward propagation pass
            loss.backward()
            
            # Execute parameter modifications
            optimizer.step()
            
            totalLoss += loss.item()
            totalRecon += reconLoss.item()
            totalKl += klLoss.item()
            
        averageLoss = totalLoss / len(trainLoader.dataset)
        averageRecon = totalRecon / len(trainLoader.dataset)
        averageKl = totalKl / len(trainLoader.dataset)
        
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {averageLoss:.4f} | Recon (MSE): {averageRecon:.4f} | KL: {averageKl:.4f}")


if __name__ == "__main__":
    trainModel()

oi
